In [5]:
import pandas as pd
import numpy as np

# data preprocessing
from sklearn import preprocessing

# exploratory analysis
import matplotlib.pyplot as plt
import mlxtend
from mlxtend.plotting import scatterplotmatrix
from mlxtend.plotting import heatmap
import seaborn as sns
from IPython.display import Image

# model fit
import statsmodels.api as sm
import tensorflow as tf


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MultiLabelBinarizer, MinMaxScaler
from sklearn.utils.class_weight import compute_class_weight

# ignore warnings (libraries are rapidly changing)
import warnings
warnings.filterwarnings('ignore')

2025-04-05 21:23:51.994165: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
###READ DATA###
df = pd.read_csv('merged_fire_data_7.csv')
print(df.columns)
df.head()

In [ ]:
############CREATE FIRE LABELS################
# Function to create forward-looking fire labels with single label selection
def create_fire_window_labels_single(df, window_size):
    # Initialize empty lists
    fire_in_window = []
    selected_conus_code = []

    for i in range(len(df)):
        # Calculate end index for the window (clamp to dataframe length)
        end_idx = min(i + window_size, len(df))

        # Check if there's a fire in the window
        window_fires = df.iloc[i:end_idx]['FIRE_START_DAY'].any()

        # Get fires in the window
        fire_data = df.iloc[i:end_idx][df.iloc[i:end_idx]['FIRE_START_DAY']]

        if len(fire_data) > 0:
            # Find the fire with largest ACRES
            # First ensure we don't have NaN values
            fire_data = fire_data[fire_data['ACRES'].notna()]

            if len(fire_data) > 0:
                # Get the fire with largest ACRES
                largest_fire = fire_data.loc[fire_data['ACRES'].idxmax()]
                selected_code = largest_fire['conus_code']

                # Only use valid conus codes
                if selected_code != 'not_used':
                    selected_conus_code.append(selected_code)
                else:
                    selected_conus_code.append('no_fire_started')
            else:
                selected_conus_code.append('no_fire_started')
        else:
            selected_conus_code.append('no_fire_started')

        # Append fire presence result
        fire_in_window.append(window_fires)

    return fire_in_window, selected_conus_code

# Create labels for each time window
for window in [3, 7, 14]:
    fires, conus_codes = create_fire_window_labels_single(df, window)
    df[f'FIRE_NEXT_{window}_DAYS'] = fires
    df[f'SELECTED_CONUS_CODE_NEXT_{window}_DAYS'] = conus_codes

# Print new columns
print(df[['FIRE_START_DAY', 'FIRE_NEXT_3_DAYS', 'SELECTED_CONUS_CODE_NEXT_3_DAYS',
          'FIRE_NEXT_7_DAYS', 'SELECTED_CONUS_CODE_NEXT_7_DAYS',
          'FIRE_NEXT_14_DAYS', 'SELECTED_CONUS_CODE_NEXT_14_DAYS']].head(10))

   FIRE_START_DAY  FIRE_NEXT_3_DAYS SELECTED_CONUS_CODE_NEXT_3_DAYS  \
0           False             False                 no_fire_started   
1           False             False                 no_fire_started   
2           False             False                 no_fire_started   
3           False             False                 no_fire_started   
4           False             False                 no_fire_started   
5           False             False                 no_fire_started   
6           False             False                 no_fire_started   
7           False             False                 no_fire_started   
8           False             False                 no_fire_started   
9           False             False                 no_fire_started   

   FIRE_NEXT_5_DAYS SELECTED_CONUS_CODE_NEXT_5_DAYS  FIRE_NEXT_7_DAYS  \
0             False                 no_fire_started             False   
1             False                 no_fire_started             False   

In [ ]:
###CREATE LAG FEATURES###
# First, ensure the DataFrame is sorted by date
df = df.sort_values(['YEAR', 'DAY_OF_YEAR']).reset_index(drop=True)

# Define the specific CONUS features to create lags for
conus_features = [
    # CONUS features (0403 suffix)
    'T2M_0403', 'T2M_MAX_0403', 'T2M_MIN_0403',
    'QV2M_0403', 'RH2M_0403', 'PRECTOTCORR_0403',
    'WS10M_0403', 'WS10M_MAX_0403', 'WS10M_MIN_0403', 'WS10M_RANGE_0403', 'WD10M_0403',

    # CONUS features (0401 suffix)
    'T2M_0401', 'T2M_MAX_0401', 'T2M_MIN_0401',
    'QV2M_0401', 'RH2M_0401', 'PRECTOTCORR_0401',
    'WS10M_0401', 'WS10M_MAX_0401', 'WS10M_MIN_0401', 'WS10M_RANGE_0401', 'WD10M_0401',

    # CONUS features (0402 suffix)
    'T2M_0402', 'T2M_MAX_0402', 'T2M_MIN_0402',
    'QV2M_0402', 'RH2M_0402', 'PRECTOTCORR_0402',
    'WS10M_0402', 'WS10M_MAX_0402', 'WS10M_MIN_0402', 'WS10M_RANGE_0402', 'WD10M_0402',

    # CONUS features (0404 suffix)
    'T2M_0404', 'T2M_MAX_0404', 'T2M_MIN_0404',
    'QV2M_0404', 'RH2M_0404', 'PRECTOTCORR_0404',
    'WS10M_0404', 'WS10M_MAX_0404', 'WS10M_MIN_0404', 'WS10M_RANGE_0404', 'WD10M_0404',

    # CONUS features (0405 suffix)
    'T2M_0405', 'T2M_MAX_0405', 'T2M_MIN_0405',
    'QV2M_0405', 'RH2M_0405', 'PRECTOTCORR_0405',
    'WS10M_0405', 'WS10M_MAX_0405', 'WS10M_MIN_0405', 'WS10M_RANGE_0405', 'WD10M_0405',

    # CONUS features (0406 suffix)
    'T2M_0406', 'T2M_MAX_0406', 'T2M_MIN_0406',
    'QV2M_0406', 'RH2M_0406', 'PRECTOTCORR_0406',
    'WS10M_0406', 'WS10M_MAX_0406', 'WS10M_MIN_0406', 'WS10M_RANGE_0406', 'WD10M_0406',

    # CONUS features (0407 suffix)
    'T2M_0407', 'T2M_MAX_0407', 'T2M_MIN_0407',
    'QV2M_0407', 'RH2M_0407', 'PRECTOTCORR_0407',
    'WS10M_0407', 'WS10M_MAX_0407', 'WS10M_MIN_0407', 'WS10M_RANGE_0407', 'WD10M_0407'
]

# Filter to only include columns that exist in the dataframe
conus_features = [col for col in conus_features if col in df.columns]
print(f"Creating lag features for {len(conus_features)} CONUS features")

# Create lag features
for lag in [3, 7, 14]: # range(2, 15, 1): #[3, 7, 14]:
    for col in conus_features:
        # Create a new column name for the lagged feature
        lag_col_name = f"{col}_LAG_{lag}"

        # Create the lagged feature
        df[lag_col_name] = df[col].shift(lag)

    print(f"Created {len(conus_features)} lag-{lag} features")

# Handle NaN values using forward fill followed by backward fill
df_filled = df.copy()
for col in df_filled.columns:
    if df_filled[col].isna().any():
        df_filled[col] = df_filled[col].fillna(method='ffill').fillna(method='bfill')

# Count NaN values before and after filling
nan_count_before = df.isna().sum().sum()
nan_count_after = df_filled.isna().sum().sum()
print(f"NaN values before filling: {nan_count_before}")
print(f"NaN values after filling: {nan_count_after}")

# Update the dataframe
df = df_filled

# Show a sample of the lagged features
lag_cols = [col for col in df.columns if 'LAG' in col]
print(f"\nSample of lag features (showing first 5 of {len(lag_cols)} columns):")
print(df[lag_cols[:5]].head(15))

In [ ]:
# Load the relabeled data
df = pd.read_csv('fire_prediction_data.csv')
# 1. Define features (using the MERRA-2 data)
# Select features for prediction

numerical_features = [
    # Original features - CONUS 0401
    'T2M_0401', 'T2M_MAX_0401', 'T2M_MIN_0401', 'QV2M_0401', 'RH2M_0401', 'PRECTOTCORR_0401',
    'WS10M_0401', 'WS10M_MAX_0401', 'WS10M_MIN_0401', 'WS10M_RANGE_0401', 'WD10M_0401',

    # Original features - CONUS 0402
    'T2M_0402', 'T2M_MAX_0402', 'T2M_MIN_0402', 'QV2M_0402', 'RH2M_0402', 'PRECTOTCORR_0402',
    'WS10M_0402', 'WS10M_MAX_0402', 'WS10M_MIN_0402', 'WS10M_RANGE_0402', 'WD10M_0402',

    # Original features - CONUS 0403
    'T2M_0403', 'T2M_MAX_0403', 'T2M_MIN_0403', 'QV2M_0403', 'RH2M_0403', 'PRECTOTCORR_0403',
    'WS10M_0403', 'WS10M_MAX_0403', 'WS10M_MIN_0403', 'WS10M_RANGE_0403', 'WD10M_0403',

    # Original features - CONUS 0404
    'T2M_0404', 'T2M_MAX_0404', 'T2M_MIN_0404', 'QV2M_0404', 'RH2M_0404', 'PRECTOTCORR_0404',
    'WS10M_0404', 'WS10M_MAX_0404', 'WS10M_MIN_0404', 'WS10M_RANGE_0404', 'WD10M_0404',

    # Original features - CONUS 0405
    'T2M_0405', 'T2M_MAX_0405', 'T2M_MIN_0405', 'QV2M_0405', 'RH2M_0405', 'PRECTOTCORR_0405',
    'WS10M_0405', 'WS10M_MAX_0405', 'WS10M_MIN_0405', 'WS10M_RANGE_0405', 'WD10M_0405',

    # Original features - CONUS 0406
    'T2M_0406', 'T2M_MAX_0406', 'T2M_MIN_0406', 'QV2M_0406', 'RH2M_0406', 'PRECTOTCORR_0406',
    'WS10M_0406', 'WS10M_MAX_0406', 'WS10M_MIN_0406', 'WS10M_RANGE_0406', 'WD10M_0406',

    # Original features - CONUS 0407
    'T2M_0407', 'T2M_MAX_0407', 'T2M_MIN_0407', 'QV2M_0407', 'RH2M_0407', 'PRECTOTCORR_0407',
    'WS10M_0407', 'WS10M_MAX_0407', 'WS10M_MIN_0407', 'WS10M_RANGE_0407', 'WD10M_0407',

    # Lagged features - 3-day lag
    'T2M_0401_LAG_3', 'T2M_MAX_0401_LAG_3', 'T2M_MIN_0401_LAG_3', 'QV2M_0401_LAG_3', 'RH2M_0401_LAG_3', 'PRECTOTCORR_0401_LAG_3',
    'WS10M_0401_LAG_3', 'WS10M_MAX_0401_LAG_3', 'WS10M_MIN_0401_LAG_3', 'WS10M_RANGE_0401_LAG_3', 'WD10M_0401_LAG_3',
    'T2M_0402_LAG_3', 'T2M_MAX_0402_LAG_3', 'T2M_MIN_0402_LAG_3', 'QV2M_0402_LAG_3', 'RH2M_0402_LAG_3', 'PRECTOTCORR_0402_LAG_3',
    'WS10M_0402_LAG_3', 'WS10M_MAX_0402_LAG_3', 'WS10M_MIN_0402_LAG_3', 'WS10M_RANGE_0402_LAG_3', 'WD10M_0402_LAG_3',
    'T2M_0403_LAG_3', 'T2M_MAX_0403_LAG_3', 'T2M_MIN_0403_LAG_3', 'QV2M_0403_LAG_3', 'RH2M_0403_LAG_3', 'PRECTOTCORR_0403_LAG_3',
    'WS10M_0403_LAG_3', 'WS10M_MAX_0403_LAG_3', 'WS10M_MIN_0403_LAG_3', 'WS10M_RANGE_0403_LAG_3', 'WD10M_0403_LAG_3',
    'T2M_0404_LAG_3', 'T2M_MAX_0404_LAG_3', 'T2M_MIN_0404_LAG_3', 'QV2M_0404_LAG_3', 'RH2M_0404_LAG_3', 'PRECTOTCORR_0404_LAG_3',
    'WS10M_0404_LAG_3', 'WS10M_MAX_0404_LAG_3', 'WS10M_MIN_0404_LAG_3', 'WS10M_RANGE_0404_LAG_3', 'WD10M_0404_LAG_3',
    'T2M_0405_LAG_3', 'T2M_MAX_0405_LAG_3', 'T2M_MIN_0405_LAG_3', 'QV2M_0405_LAG_3', 'RH2M_0405_LAG_3', 'PRECTOTCORR_0405_LAG_3',
    'WS10M_0405_LAG_3', 'WS10M_MAX_0405_LAG_3', 'WS10M_MIN_0405_LAG_3', 'WS10M_RANGE_0405_LAG_3', 'WD10M_0405_LAG_3',
    'T2M_0406_LAG_3', 'T2M_MAX_0406_LAG_3', 'T2M_MIN_0406_LAG_3', 'QV2M_0406_LAG_3', 'RH2M_0406_LAG_3', 'PRECTOTCORR_0406_LAG_3',
    'WS10M_0406_LAG_3', 'WS10M_MAX_0406_LAG_3', 'WS10M_MIN_0406_LAG_3', 'WS10M_RANGE_0406_LAG_3', 'WD10M_0406_LAG_3',
    'T2M_0407_LAG_3', 'T2M_MAX_0407_LAG_3', 'T2M_MIN_0407_LAG_3', 'QV2M_0407_LAG_3', 'RH2M_0407_LAG_3', 'PRECTOTCORR_0407_LAG_3',
    'WS10M_0407_LAG_3', 'WS10M_MAX_0407_LAG_3', 'WS10M_MIN_0407_LAG_3', 'WS10M_RANGE_0407_LAG_3', 'WD10M_0407_LAG_3',

    # Lagged features - 7-day lag
    'T2M_0401_LAG_7', 'T2M_MAX_0401_LAG_7', 'T2M_MIN_0401_LAG_7', 'QV2M_0401_LAG_7', 'RH2M_0401_LAG_7', 'PRECTOTCORR_0401_LAG_7',
    'WS10M_0401_LAG_7', 'WS10M_MAX_0401_LAG_7', 'WS10M_MIN_0401_LAG_7', 'WS10M_RANGE_0401_LAG_7', 'WD10M_0401_LAG_7',
    'T2M_0402_LAG_7', 'T2M_MAX_0402_LAG_7', 'T2M_MIN_0402_LAG_7', 'QV2M_0402_LAG_7', 'RH2M_0402_LAG_7', 'PRECTOTCORR_0402_LAG_7',
    'WS10M_0402_LAG_7', 'WS10M_MAX_0402_LAG_7', 'WS10M_MIN_0402_LAG_7', 'WS10M_RANGE_0402_LAG_7', 'WD10M_0402_LAG_7',
    'T2M_0403_LAG_7', 'T2M_MAX_0403_LAG_7', 'T2M_MIN_0403_LAG_7', 'QV2M_0403_LAG_7', 'RH2M_0403_LAG_7', 'PRECTOTCORR_0403_LAG_7',
    'WS10M_0403_LAG_7', 'WS10M_MAX_0403_LAG_7', 'WS10M_MIN_0403_LAG_7', 'WS10M_RANGE_0403_LAG_7', 'WD10M_0403_LAG_7',
    'T2M_0404_LAG_7', 'T2M_MAX_0404_LAG_7', 'T2M_MIN_0404_LAG_7', 'QV2M_0404_LAG_7', 'RH2M_0404_LAG_7', 'PRECTOTCORR_0404_LAG_7',
    'WS10M_0404_LAG_7', 'WS10M_MAX_0404_LAG_7', 'WS10M_MIN_0404_LAG_7', 'WS10M_RANGE_0404_LAG_7', 'WD10M_0404_LAG_7',
    'T2M_0405_LAG_7', 'T2M_MAX_0405_LAG_7', 'T2M_MIN_0405_LAG_7', 'QV2M_0405_LAG_7', 'RH2M_0405_LAG_7', 'PRECTOTCORR_0405_LAG_7',
    'WS10M_0405_LAG_7', 'WS10M_MAX_0405_LAG_7', 'WS10M_MIN_0405_LAG_7', 'WS10M_RANGE_0405_LAG_7', 'WD10M_0405_LAG_7',
    'T2M_0406_LAG_7', 'T2M_MAX_0406_LAG_7', 'T2M_MIN_0406_LAG_7', 'QV2M_0406_LAG_7', 'RH2M_0406_LAG_7', 'PRECTOTCORR_0406_LAG_7',
    'WS10M_0406_LAG_7', 'WS10M_MAX_0406_LAG_7', 'WS10M_MIN_0406_LAG_7', 'WS10M_RANGE_0406_LAG_7', 'WD10M_0406_LAG_7',
    'T2M_0407_LAG_7', 'T2M_MAX_0407_LAG_7', 'T2M_MIN_0407_LAG_7', 'QV2M_0407_LAG_7', 'RH2M_0407_LAG_7', 'PRECTOTCORR_0407_LAG_7',
    'WS10M_0407_LAG_7', 'WS10M_MAX_0407_LAG_7', 'WS10M_MIN_0407_LAG_7', 'WS10M_RANGE_0407_LAG_7', 'WD10M_0407_LAG_7',

    # Lagged features - 14-day lag
    'T2M_0401_LAG_14', 'T2M_MAX_0401_LAG_14', 'T2M_MIN_0401_LAG_14', 'QV2M_0401_LAG_14', 'RH2M_0401_LAG_14', 'PRECTOTCORR_0401_LAG_14',
    'WS10M_0401_LAG_14', 'WS10M_MAX_0401_LAG_14', 'WS10M_MIN_0401_LAG_14', 'WS10M_RANGE_0401_LAG_14', 'WD10M_0401_LAG_14',
    'T2M_0402_LAG_14', 'T2M_MAX_0402_LAG_14', 'T2M_MIN_0402_LAG_14', 'QV2M_0402_LAG_14', 'RH2M_0402_LAG_14', 'PRECTOTCORR_0402_LAG_14',
    'WS10M_0402_LAG_14', 'WS10M_MAX_0402_LAG_14', 'WS10M_MIN_0402_LAG_14', 'WS10M_RANGE_0402_LAG_14', 'WD10M_0402_LAG_14',
    'T2M_0403_LAG_14', 'T2M_MAX_0403_LAG_14', 'T2M_MIN_0403_LAG_14', 'QV2M_0403_LAG_14', 'RH2M_0403_LAG_14', 'PRECTOTCORR_0403_LAG_14',
    'WS10M_0403_LAG_14', 'WS10M_MAX_0403_LAG_14', 'WS10M_MIN_0403_LAG_14', 'WS10M_RANGE_0403_LAG_14', 'WD10M_0403_LAG_14',
    'T2M_0404_LAG_14', 'T2M_MAX_0404_LAG_14', 'T2M_MIN_0404_LAG_14', 'QV2M_0404_LAG_14', 'RH2M_0404_LAG_14', 'PRECTOTCORR_0404_LAG_14',
    'WS10M_0404_LAG_14', 'WS10M_MAX_0404_LAG_14', 'WS10M_MIN_0404_LAG_14', 'WS10M_RANGE_0404_LAG_14', 'WD10M_0404_LAG_14',
    'T2M_0405_LAG_14', 'T2M_MAX_0405_LAG_14', 'T2M_MIN_0405_LAG_14', 'QV2M_0405_LAG_14', 'RH2M_0405_LAG_14', 'PRECTOTCORR_0405_LAG_14',
    'WS10M_0405_LAG_14', 'WS10M_MAX_0405_LAG_14', 'WS10M_MIN_0405_LAG_14', 'WS10M_RANGE_0405_LAG_14', 'WD10M_0405_LAG_14',
    'T2M_0406_LAG_14', 'T2M_MAX_0406_LAG_14', 'T2M_MIN_0406_LAG_14', 'QV2M_0406_LAG_14', 'RH2M_0406_LAG_14', 'PRECTOTCORR_0406_LAG_14',
    'WS10M_0406_LAG_14', 'WS10M_MAX_0406_LAG_14', 'WS10M_MIN_0406_LAG_14', 'WS10M_RANGE_0406_LAG_14', 'WD10M_0406_LAG_14',
    'T2M_0407_LAG_14', 'T2M_MAX_0407_LAG_14', 'T2M_MIN_0407_LAG_14', 'QV2M_0407_LAG_14', 'RH2M_0407_LAG_14', 'PRECTOTCORR_0407_LAG_14',
    'WS10M_0407_LAG_14', 'WS10M_MAX_0407_LAG_14', 'WS10M_MIN_0407_LAG_14', 'WS10M_RANGE_0407_LAG_14', 'WD10M_0407_LAG_14'
]


# Remove any features that don't exist in the DataFrame
numerical_features = [f for f in numerical_features if f in df.columns]



# Add cyclical features
df['MONTH_SIN'] = np.sin(2 * np.pi * df['MONTH'] / 12)
df['MONTH_COS'] = np.cos(2 * np.pi * df['MONTH'] / 12)
df['DAY_SIN'] = np.sin(2 * np.pi * df['DAY_OF_YEAR'] / 365)
df['DAY_COS'] = np.cos(2 * np.pi * df['DAY_OF_YEAR'] / 365)

# One-hot encode season
season_dummies = pd.get_dummies(df['SEASON'], prefix='SEASON')
df = pd.concat([df, season_dummies], axis=1)

# One-hot encode cause
cause_dummies = pd.get_dummies(df['CAUSE'], prefix='CAUSE')
df = pd.concat([df, cause_dummies], axis=1)

# One-hot encode conus codes
conus_codes_dummies = pd.get_dummies(df['conus_code'], prefix='conus_code')
df = pd.concat([df, conus_codes_dummies], axis=1)

# Feature list
features = numerical_features + ['MONTH_SIN', 'MONTH_COS', 'DAY_SIN', 'DAY_COS'] + list(season_dummies.columns) + list(cause_dummies.columns) + list(conus_codes_dummies.columns)

# Create a mapping of conus codes to indices
all_conus_codes = df['SELECTED_CONUS_CODE_NEXT_3_DAYS'].unique()
conus_code_to_index = {code: idx for idx, code in enumerate(all_conus_codes)}

# Convert labels to indices
y_3 = df['SELECTED_CONUS_CODE_NEXT_3_DAYS'].map(conus_code_to_index).values
y_7 = df['SELECTED_CONUS_CODE_NEXT_7_DAYS'].map(conus_code_to_index).values
y_14 = df['SELECTED_CONUS_CODE_NEXT_14_DAYS'].map(conus_code_to_index).values


# Prepare features
X = df[features].copy()

# Train-validation-test split
X_temp, X_test, y_temp, y_test_3 = train_test_split(X, y_3, test_size=0.2, random_state=42)
X_train, X_val, y_train_3, y_val = train_test_split(X_temp, y_temp, test_size=0.25)

X_temp, X_test, y_temp, y_test_7 = train_test_split(X, y_7, test_size=0.2, random_state=42)
X_train, X_val, y_train_7, y_val = train_test_split(X_temp, y_temp, test_size=0.25)

X_temp, X_test, y_temp, y_test_14 = train_test_split(X, y_14, test_size=0.2, random_state=42)
X_train, X_val, y_train_14, y_val = train_test_split(X_temp, y_temp, test_size=0.25)
'''

# 5. Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)
'''
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [ ]:
####SAVE PREPROCESSED DATAFRAME####
# Save the DataFrame with all target labels to CSV
df.to_csv('fire_prediction_data.csv', index=False)